In [25]:
import re
import jieba
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 全局设备 & 超参数
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"运行设备: {device}")

# 序列标记
SOS = "<sos>"
EOS = "<eos>"
PAD = "<pad>"
UNK = "<unk>"

# 超参数（降低轮数防过拟合）
BATCH_SIZE = 8
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
EPOCHS = 8
LEARNING_RATE = 1e-3
MAX_SEQ_LEN = 20

运行设备: cpu


In [26]:
# 文本清洗
def clean_text(text):
    text = re.sub(r"[^\u4e00-\u9fa5a-zA-Z0-9\s]", "", text)
    return text.strip()

# 中文分词 + 英文分词
def tokenize_zh(text):
    return list(jieba.cut(clean_text(text)))

def tokenize_en(text):
    return clean_text(text).lower().split()

# 加载平行语料（过滤空行、空句子）
def load_corpus(file_path):
    src_list, tgt_list = [], []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or "\t" not in line:
                continue
            zh, en = line.split("\t", 1)
            zh = zh.strip()
            en = en.strip()
            if not zh or not en:
                continue
            # 分词 + 起止符
            src_tokens = [SOS] + tokenize_zh(zh) + [EOS]
            tgt_tokens = [SOS] + tokenize_en(en) + [EOS]
            src_list.append(src_tokens[:MAX_SEQ_LEN])
            tgt_list.append(tgt_tokens[:MAX_SEQ_LEN])
    return src_list, tgt_list

# 构建词表
class Vocab:
    def __init__(self):
        self.word2idx = {PAD:0, SOS:1, EOS:2, UNK:3}
        self.idx2word = {0:PAD, 1:SOS, 2:EOS, 3:UNK}
        self.vocab_size = 4

    def add_sentence(self, tokens):
        for word in tokens:
            if word not in self.word2idx:
                self.word2idx[word] = self.vocab_size
                self.idx2word[self.vocab_size] = word
                self.vocab_size += 1

    def sentence2idx(self, tokens):
        return [self.word2idx.get(w, 3) for w in tokens]

# 自定义数据集（强制 long 类型）
class TransDataset(Dataset):
    def __init__(self, src_data, tgt_data, src_vocab, tgt_vocab):
        self.src_data = src_data
        self.tgt_data = tgt_data
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        src = self.src_vocab.sentence2idx(self.src_data[idx])
        tgt = self.tgt_vocab.sentence2idx(self.tgt_data[idx])
        # 填充
        src += [0] * (MAX_SEQ_LEN - len(src))
        tgt += [0] * (MAX_SEQ_LEN - len(tgt))
        # 强制转为 LongTensor，适配Embedding
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

# 加载数据集
train_src, train_tgt = load_corpus("data/train.txt")
val_src, val_tgt = load_corpus("data/val.txt")
test_src, test_tgt = load_corpus("data/test.txt")

print(f"训练集条数: {len(train_src)}")
print(f"验证集条数: {len(val_src)}")
print(f"测试集条数: {len(test_src)}")

# 构建词表
src_vocab = Vocab()
tgt_vocab = Vocab()
for s, t in zip(train_src, train_tgt):
    src_vocab.add_sentence(s)
    tgt_vocab.add_sentence(t)

print(f"中文词表大小: {src_vocab.vocab_size}")
print(f"英文词表大小: {tgt_vocab.vocab_size}")

# 构建DataLoader
train_dataset = TransDataset(train_src, train_tgt, src_vocab, tgt_vocab)
val_dataset = TransDataset(val_src, val_tgt, src_vocab, tgt_vocab)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

训练集条数: 1127
验证集条数: 216
测试集条数: 220
中文词表大小: 2581
英文词表大小: 1937


In [29]:
# 标准Bahdanau注意力机制（适配LSTM双层结构）
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_outputs):
        # enc_outputs: [seq_len, batch, hidden_dim]
        seq_len, batch, _ = enc_outputs.shape
        # 取解码器最后一层隐藏态: [num_layers, batch, hidden] -> [batch, hidden]
        dec_h = dec_hidden[-1]
        # 扩展维度对齐序列长度
        dec_h = dec_h.unsqueeze(0).repeat(seq_len, 1, 1)

        # 拼接: [seq_len, batch, 2*hidden]
        concat = torch.cat([dec_h, enc_outputs], dim=-1)
        energy = torch.tanh(self.attn(concat))
        # 注意力权重 [seq_len, batch, 1]
        attn_score = self.v(energy)
        attn_weight = torch.softmax(attn_score, dim=0)
        # 加权求和得到上下文向量 [1, batch, hidden]
        context = torch.sum(attn_weight * enc_outputs, dim=0, keepdim=True)
        return context, attn_weight.squeeze(-1)


# 编码器 双层LSTM
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=False)

    def forward(self, src):
        embed = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embed)
        return outputs, hidden, cell


# 解码器 双层LSTM + 注意力
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = Attention(hidden_dim)
        self.lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim, num_layers, batch_first=False)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_tok, dec_hidden, dec_cell, enc_outputs):
        # input_tok: [1, batch]
        embed = self.embedding(input_tok)  # [1, batch, embed_dim]
        # 注意力计算
        context, attn_w = self.attention(dec_hidden, enc_outputs)
        # 拼接词嵌入 + 上下文向量
        lstm_in = torch.cat([embed, context], dim=-1)
        # LSTM前向
        output, (dec_hidden, dec_cell) = self.lstm(lstm_in, (dec_hidden, dec_cell))
        pred = self.fc(output)
        return pred, dec_hidden, dec_cell, attn_w


# 整体Seq2Seq模型
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.7):
        tgt_len, batch = tgt.shape
        tgt_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(tgt_len, batch, tgt_vocab_size).to(device)

        enc_outputs, enc_hidden, enc_cell = self.encoder(src)
        dec_hidden, dec_cell = enc_hidden, enc_cell
        input_tok = tgt[0:1, :]

        for t_step in range(1, tgt_len):
            output, dec_hidden, dec_cell, _ = self.decoder(input_tok, dec_hidden, dec_cell, enc_outputs)
            outputs[t_step] = output
            # 教师强制
            teacher_force = np.random.random() < teacher_forcing_ratio
            top1 = output.argmax(dim=-1)
            input_tok = tgt[t_step:t_step+1, :] if teacher_force else top1
        return outputs


# 初始化模型
encoder = Encoder(src_vocab.vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
decoder = Decoder(tgt_vocab.vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
model = Seq2Seq(encoder, decoder).to(device)

# 损失函数、优化器
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("模型初始化完成")

模型初始化完成


In [30]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for src, tgt in tqdm(loader):
        src = src.transpose(0,1).to(device)
        tgt = tgt.transpose(0,1).to(device)
        optimizer.zero_grad()
        output = model(src, tgt)
        loss = criterion(output.reshape(-1, output.shape[-1]), tgt.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src = src.transpose(0,1).to(device)
            tgt = tgt.transpose(0,1).to(device)
            output = model(src, tgt, teacher_forcing_ratio=0)
            loss = criterion(output.reshape(-1, output.shape[-1]), tgt.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(loader)

# 开始训练
train_loss_list = []
val_loss_list = []
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss = val_epoch(model, val_loader, criterion)
    train_loss_list.append(train_loss)
    val_loss_list.append(val_loss)
    print(f"Epoch {epoch+1:2d} | 训练损失: {train_loss:.4f} | 验证损失: {val_loss:.4f}")

# 绘制损失曲线
plt.figure(figsize=(10,5))
plt.plot(train_loss_list, label="Train Loss")
plt.plot(val_loss_list, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("训练&验证损失曲线")
plt.legend()
plt.savefig("loss_curve.png", dpi=300, bbox_inches="tight")
plt.show()

100%|████████████████████████████████████████████████████████████████████████████████| 141/141 [00:45<00:00,  3.07it/s]


Epoch  1 | 训练损失: 6.4048 | 验证损失: 6.2903


100%|████████████████████████████████████████████████████████████████████████████████| 141/141 [00:45<00:00,  3.09it/s]


Epoch  2 | 训练损失: 5.9158 | 验证损失: 6.3493


100%|████████████████████████████████████████████████████████████████████████████████| 141/141 [00:45<00:00,  3.09it/s]


...

100%|████████████████████████████████████████████████████████████████████████████████| 141/141 [00:45<00:00,  3.12it/s]


Epoch  7 | 训练损失: 5.0909 | 验证损失: 6.6028


100%|████████████████████████████████████████████████████████████████████████████████| 141/141 [00:44<00:00,  3.15it/s]


Epoch  8 | 训练损失: 4.9472 | 验证损失: 6.5813


C:\Users\wxf88\AppData\Local\Temp\ipykernel_37900\3022360703.py:46: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [33]:
def translate_sentence(model, src_sent, src_vocab, tgt_vocab):
    model.eval()
    raw_tokens = [SOS] + tokenize_zh(src_sent) + [EOS]
    if len(raw_tokens) < 2:
        return [UNK, EOS], np.array([]), raw_tokens
    src_idx = src_vocab.sentence2idx(raw_tokens)[:MAX_SEQ_LEN]
    src_tensor = torch.tensor(src_idx, dtype=torch.long).unsqueeze(1).to(device)

    with torch.no_grad():
        enc_outputs, enc_hidden, enc_cell = model.encoder(src_tensor)
        dec_hidden, dec_cell = enc_hidden, enc_cell
        input_tok = torch.tensor([[src_vocab.word2idx[SOS]]], dtype=torch.long).to(device)

        pred_tokens = []
        attn_matrix = []

        for _ in range(MAX_SEQ_LEN):
            output, dec_hidden, dec_cell, attn_w = model.decoder(input_tok, dec_hidden, dec_cell, enc_outputs)
            attn_matrix.append(attn_w.cpu().numpy().squeeze())
            top1 = output.argmax(dim=-1)
            word_idx = top1.item()

            if word_idx in tgt_vocab.idx2word:
                word = tgt_vocab.idx2word[word_idx]
            else:
                word = UNK

            pred_tokens.append(word)
            if word == EOS:
                break
            input_tok = top1

    return pred_tokens, np.array(attn_matrix), raw_tokens

# 计算BLEU分数
def calc_bleu(test_src_list, test_tgt_list, model, src_vocab, tgt_vocab):
    smoothie = SmoothingFunction().method4
    total_bleu = 0
    count = 0
    for src_tokens, tgt_tokens in zip(test_src_list, test_tgt_list):
        src_sent = "".join([w for w in src_tokens if w not in [SOS,EOS]])
        if not src_sent:
            continue
        ref = [[w for w in tgt_tokens if w not in [SOS,EOS,PAD]]]
        pred, _, _ = translate_sentence(model, src_sent, src_vocab, tgt_vocab)
        pred = [w for w in pred if w not in [SOS,EOS,PAD]]
        if len(pred) == 0 or len(ref[0]) == 0:
            continue
        total_bleu += sentence_bleu(ref, pred, smoothing_function=smoothie)
        count += 1
    if count == 0:
        return 0.0
    return total_bleu / count

# 测试集BLEU分数
bleu_score = calc_bleu(test_src, test_tgt, model, src_vocab, tgt_vocab)
print(f"测试集平均BLEU分数: {bleu_score:.4f}")

# 注意力可视化
test_zh = "我喜欢自然语言处理"
pred_en, attn_map, src_tokens = translate_sentence(model, test_zh, src_vocab, tgt_vocab)
print("源句子(中文):", test_zh)
print("翻译结果(英文):", " ".join(pred_en))

# 绘制热力图（增加空值判断，防止报错）
if attn_map.ndim > 0 and attn_map.size > 0:
    plt.figure(figsize=(12,6))
    sns.heatmap(attn_map, xticklabels=src_tokens, yticklabels=pred_en, cmap="YlGnBu")
    plt.xlabel("中文源词")
    plt.ylabel("英文目标词")
    plt.title("注意力权重对齐图")
    plt.savefig("attention_vis.png", dpi=300, bbox_inches="tight")
    plt.show()

测试集平均BLEU分数: 0.0089
源句子(中文): 我喜欢自然语言处理
翻译结果(英文): town <eos>


C:\Users\wxf88\AppData\Local\Temp\ipykernel_37900\2839758369.py:73: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [34]:
# 测试案例集合
test_cases = [
    "今天天气很好",
    "我不喜欢下雨天",
    "人工智能改变了我们的生活",
    "他每天都会阅读书籍",
    "一帆风顺"
]

for case in test_cases:
    res, _, _ = translate_sentence(model, case, src_vocab, tgt_vocab)
    print(f"原文：{case}")
    print(f"译文：{' '.join(res)}")
    print("-"*50)

原文：今天天气很好
译文：<eos>
--------------------------------------------------
原文：我不喜欢下雨天
译文：sick my <eos>
--------------------------------------------------
原文：人工智能改变了我们的生活
译文：the the <eos>
--------------------------------------------------
原文：他每天都会阅读书籍
译文：the <eos>
--------------------------------------------------
原文：一帆风顺
译文：<eos>
--------------------------------------------------
